In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_csv(r"C:\Users\KIIT0001\Downloads\flood_risk_dataset_corrected.csv")
data

,Latitude,Longitude,Rainfall (mm),Temperature (°C),Humidity (%),River Discharge (m³/s),Water Level (m),Elevation (m),Land Cover,Soil Type,Population Density,Infrastructure,Historical Floods,Flood Occurred
0,18.861663,78.835584,218.999493,34.144337,43.912963,4236.182888,7.415552,377.465433,Water Body,Clay,7276.742184,1,0,1
1,35.570715,77.654451,55.353599,28.778774,27.585422,2472.585219,8.811019,7330.608875,Forest,Peat,6897.736956,0,1,0
2,29.227824,73.108463,103.991908,43.934956,30.108738,977.328053,4.631799,2205.873488,Agricultural,Loam,4361.518494,1,1,0
3,25.361096,85.610733,198.984191,21.569354,34.453690,3683.208933,2.891787,2512.277800,Desert,Sandy,6163.069701,1,1,1
4,12.524541,81.822101,144.626803,32.635692,36.292267,2093.390678,3.188466,2001.818223,Agricultural,Loam,6167.964591,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,32.872024,93.434120,285.682635,37.621017,69.795616,4830.703665,5.943965,2850.197900,Agricultural,Clay,6943.559433,0,0,0
9996,34.027756,69.357605,224.347263,37.935808,38.095486,1866.199787,9.466158,3314.692947,Forest,Clay,3011.997459,1,0,0
9997,35.454530,76.807256,5.836759,23.087083,79.919607,1523.374305,9.209185,3377.296962,Desert,Clay,7149.938303,1,0,0
9998,19.527152,80.856280,120.301453,28.029593,61.680873,2036.812638,2.004644,1146.986151,Water Body,Sandy,906.031452,1,0,0


In [3]:
data.isnull().sum

<bound method DataFrame.sum of       Latitude  Longitude  Rainfall (mm)  Temperature (°C)  Humidity (%)  \
0        False      False          False             False         False   
1        False      False          False             False         False   
2        False      False          False             False         False   
3        False      False          False             False         False   
4        False      False          False             False         False   
...        ...        ...            ...               ...           ...   
9995     False      False          False             False         False   
9996     False      False          False             False         False   
9997     False      False          False             False         False   
9998     False      False          False             False         False   
9999     False      False          False             False         False   

      River Discharge (m³/s)  Water Level (m)  Elevation

In [4]:
data['Flood Occurred'].value_counts()

Flood Occurred
0    7655
1    2345
Name: count, dtype: int64

In [5]:
#Encoding categorical columns
#Land Cover

from sklearn.preprocessing import LabelEncoder

le_land = LabelEncoder()

data['Land Cover'] = le_land.fit_transform(data['Land Cover'])

print(dict(zip(le_land.classes_,le_land.transform(le_land.classes_))))

{'Agricultural': 0, 'Desert': 1, 'Forest': 2, 'Urban': 3, 'Water Body': 4}


In [6]:
#Soil type encoding

le_soil = LabelEncoder()

data['Soil Type'] = le_soil.fit_transform(data['Soil Type'])

print(dict(zip(le_soil.classes_,le_soil.transform(le_soil.classes_))))

{'Clay': 0, 'Loam': 1, 'Peat': 2, 'Sandy': 3, 'Silt': 4}


In [7]:
features = ['Latitude','Longitude','Rainfall (mm)','Temperature (°C)','Humidity (%)',
           'River Discharge (m³/s)','Water Level (m)','Elevation (m)','Population Density',
            'Infrastructure','Historical Floods','Land Cover','Soil Type']

target = 'Flood Occurred'

In [8]:
x = data[features]
y = data[target]

In [9]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state = 42)

## Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42)
rf.fit(x_train, y_train)

RandomForestClassifier(max_depth=5, random_state=42)

In [17]:
y_pred = rf.predict(x_test)
y_pred

array([0, 1, 0, ..., 0, 0, 0], dtype=int64)

In [18]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

acc = accuracy_score(y_pred,y_test)
re = recall_score(y_pred,y_test)
pr = precision_score(y_pred,y_test)
f1 = f1_score(y_pred,y_test)

print('Accuracy:',acc)
print('Recall:',re)
print('Precision:',pr)
print('F1-Score:',f1)

Accuracy: 0.9315
Recall: 1.0
Precision: 0.7078891257995735
F1-Score: 0.8289637952559301


In [19]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(y_pred,y_test)
cm

array([[1531,  137],
       [   0,  332]], dtype=int64)

## XGBoost

In [14]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=5,random_state=42)
xgb.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [15]:
y_pred2 = xgb.predict(x_test)
y_pred2

array([0, 1, 0, ..., 0, 0, 0])

In [16]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

acc = accuracy_score(y_pred2,y_test)
re = recall_score(y_pred2,y_test)
pr = precision_score(y_pred2,y_test)
f1 = f1_score(y_pred2,y_test)

print('Accuracy:',acc)
print('Recall:',re)
print('Precision:',pr)
print('F1-Score:',f1)

Accuracy: 0.999
Recall: 1.0
Precision: 0.9957356076759062
F1-Score: 0.9978632478632479
